## Getting Reddit Threads for Each K-Drama and Consolidating into Master Dataset
Author: Kelly Chen

**Table of Contents**
1. [Review Extracted Redit Threads of K-Dramas](#sec1)
2. [Consolidate Threads Into One Full Dataset](#sec2)

GenAI was used to provide inputs on how to consolidate extracted thread files together.

<a id="sec1"></a>

## 1. Review Extracted Redit Threads of K-Dramas
We begin putting together our dataset by finding all Reddit threads or comments that mentions our specific K-dramas we chose to focus on for analyzing beauty standards, which includes:
- *Mask Girl* (2023)
- *True Beauty* (2020)
- *My ID is Gangnam Beauty* (2018)
- *Birth of a Beauty* (2014)
- *Boys Over Flowers* (2009)

First, we will use the Reddit Search Engine Streamlit to find the threads/comments that contain the name of our K-dramas and download the metadata as a jsonl file. Once we downloaded the jsonl files containing all threads, we will next extract the content using the extract_thread_bodies.py file, which contains code to search for thread ids and return their content. This step is done; below, we will make sure the threads are extracted correctly.

Check how many threads are present for each K-drama:

In [ ]:
import os

In [26]:
BASE_DIR = "exported_threads"

for drama in os.listdir(BASE_DIR):
    drama_path = os.path.join(BASE_DIR, drama) #get specific path for each drama

    if os.path.isdir(drama_path):
        files = [f for f in os.listdir(drama_path) if f.endswith(".jsonl")]
        print(f"{drama}: {len(files)} threads") #get length

threads_mask_girl: 1533 threads
threads_true_beauty: 9107 threads
threads_my_id_is_gangnam_beauty: 3628 threads
threads_birth_of_a_beauty: 1033 threads
threads_boys_over_flowers: 6881 threads


View one individual thread to see what details are there:

In [1]:
import json

Inspect a thread from *True Beauty*:

In [28]:
file_path = "exported_threads/threads_true_beauty/t3_1afoclk.jsonl" #randomly selected thread from True Beauty

with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i == 3: #print 3 records
            break
        record = json.loads(line)
        print(json.dumps(record, indent=2, ensure_ascii=False))

{
  "timestamp": "1706724659",
  "subreddit": "kdramarecommends",
  "title": "Sweet drama",
  "selftext": "I want a high school sweet lovely kdrama something like : cheer-up ... extraordinary \nWhere the female lead is funny and energetic",
  "url": "https://www.reddit.com/r/kdramarecommends/comments/1afoclk/sweet_drama/",
  "score": 1,
  "num_comments": 1,
  "permalink": "https://reddit.com/r/kdramarecommends/comments/1afoclk/sweet_drama/",
  "post_id": "1afoclk",
  "full_id": "t3_1afoclk",
  "flair": "Recommendation Request",
  "author": "user_7f6fceec9456"
}
{
  "comment_id": "kod7dhr",
  "post_id": "t3_1afoclk",
  "parent_id": "t3_1afoclk",
  "subreddit": "kdramarecommends",
  "comment_body": "1. True Beauty\n2. 2521\n3. A love so beautiful",
  "created_utc": "1706748311.0",
  "score": 3.0,
  "ups": 3.0,
  "downs": 0,
  "controversiality": 0,
  "gilded": 0,
  "total_awards": 0,
  "distinguished": null,
  "edited": "false",
  "author": "user_1ff1d448b305"
}
{
  "comment_id": "kojvk3

Structure looks good! These threads/comments all mention *True Beauty* in the text body.

<a id="sec2"></a>

## 2. Consolidate Threads Into One Full Dataset
Now that we have verified the extracted thread content, we can consolidate all of our extracted threads from each K-drama into one big dataset so that we can use it for further analysis like topic modeling to identify common themes. 

In [24]:
import pandas as pd

Below code was outputed by AI and validated and commented on by me.

In [ ]:
BASE_DIR = "exported_threads"

all_records = []

for drama in os.listdir(BASE_DIR):

    drama_path = os.path.join(BASE_DIR, drama) #get drama path for each drama

    if not os.path.isdir(drama_path):
        continue

    for filename in os.listdir(drama_path):

        if not filename.endswith(".jsonl"): #ensure jsonl file
            continue

        file_path = os.path.join(drama_path, filename) #get path for thread

        thread_id = filename.replace(".jsonl", "") #extracts id from thread name
        thread_id = thread_id.removeprefix("t3_") #removes "t3" from beginning of file name to get just id

        with open(file_path, "r", encoding="utf-8") as f:

            for line in f:
                if not line.strip():
                    continue

                record = json.loads(line)

                # Identify submission vs comment
                if "comment_id" in record:
                    record_type = "comment"
                    text = record.get("comment_body", "")
                    record_id = record.get("comment_id")
                    timestamp = record.get("created_utc")

                else:
                    record_type = "submission"
                    text = record.get("selftext", "")
                    record_id = record.get("post_id")
                    timestamp = record.get("timestamp")

                # Standardize record
                all_records.append({
                    "drama": drama.replace("threads_", "").replace("_", " ").title(), #get only the drama name and format as title
                    "thread_id": thread_id,
                    "record_type": record_type,
                    "record_id": record_id,
                    "parent_id": record.get("parent_id"),
                    "title": record.get("title"),
                    "text": text,
                    "subreddit": record.get("subreddit"),
                    "timestamp": timestamp,
                    "score": record.get("score"),
                    "author": record.get("author"),
                    "url": record.get("url"),
                    "num_comments": record.get("num_comments")
                })

In [33]:
df = pd.DataFrame(all_records) #create dataframe
df.head(10)

,drama,thread_id,record_type,record_id,parent_id,title,text,subreddit,timestamp,score,author,url,num_comments
0,Mask Girl,18uumbs,submission,18uumbs,None,r/KDRAMA Challenge 2023 - December Check In - ...,[Hello everyone!](https://i.imgur.com/XDaA7zV....,KDRAMA,1703983061.0,32.0,user_8cbff80d5226,https://www.reddit.com/r/KDRAMA/comments/18uum...,51.0
1,Mask Girl,18uumbs,comment,kfmq9dn,t3_18uumbs,None,I managed to finish my challenge with around 3...,KDRAMA,1703983121.0,10.0,user_8cbff80d5226,None,NaN
2,Mask Girl,18uumbs,comment,kfmw6s3,t1_kfmq9dn,None,Congrats on your chaebol status!! Also thank y...,KDRAMA,1703985478.0,6.0,user_70f9ec7a84d2,None,NaN
3,Mask Girl,18uumbs,comment,kfmzkcd,t3_18uumbs,None,I've been waiting very excitedly for this post...,KDRAMA,1703986787.0,8.0,user_70f9ec7a84d2,None,NaN
4,Mask Girl,18uumbs,comment,kfn2te2,t3_18uumbs,None,I finished my challenge back in October? Novem...,KDRAMA,1703988041.0,24.0,user_6be091ed4476,None,NaN
5,Mask Girl,18uumbs,comment,kfn4xj3,t3_18uumbs,None,Will be participating for the first time in 20...,KDRAMA,1703988873.0,7.0,user_436a6c797c20,None,NaN
6,Mask Girl,18uumbs,comment,kfn5b1a,t3_18uumbs,None,Completed Dec 26. Chaebol status achieved. In ...,KDRAMA,1703989026.0,5.0,user_1aa399cdfa57,None,NaN
7,Mask Girl,18uumbs,comment,kfn5fuj,t3_18uumbs,None,Thank you for organizing!!! Completed chaebol ...,KDRAMA,1703989079,5.0,user_d5330e14046f,None,NaN
8,Mask Girl,18uumbs,comment,kfn5p3h,t3_18uumbs,None,"Thanks for another excellent challenge year, y...",KDRAMA,1703989178.0,9.0,user_a71331b04b59,None,NaN
9,Mask Girl,18uumbs,comment,kfn6ag2,t1_kfn2te2,None,The challenge is addictive isn’t it. I’m looki...,KDRAMA,1703989416.0,6.0,user_563bf806a40e,None,NaN


In [34]:
df.tail(10)

,drama,thread_id,record_type,record_id,parent_id,title,text,subreddit,timestamp,score,author,url,num_comments
1940029,Boys Over Flowers,ids296,comment,g2fgwal,t3_ids296,None,I watched this series called **Promised Neverl...,KDRAMA,1598081281,2.0,user_fa140cd03568,None,NaN
1940030,Boys Over Flowers,ids296,comment,g2gd4nj,t3_ids296,None,After a two-year hiatus the [Korean Drama Podc...,KDRAMA,1598097550,3.0,user_ceb2709c0695,None,NaN
1940031,Boys Over Flowers,ids296,comment,g2gp3ee,t1_g2eo69l,None,"Oooh, thanks! I'm looking forward to experienc...",KDRAMA,1598102486,1.0,user_9b891ee16b5d,None,NaN
1940032,Boys Over Flowers,ids296,comment,g2gp5lu,t1_g2d4977,None,"I haven't yet, thank you for the recommendatio...",KDRAMA,1598102512,1.0,user_9b891ee16b5d,None,NaN
1940033,Boys Over Flowers,ids296,comment,g2hst52,t3_ids296,None,What is so good about I Live Alone? I’m just w...,KDRAMA,1598117224,2.0,user_d070b4c03ab0,None,NaN
1940034,Boys Over Flowers,ids296,comment,g2im2ln,t1_g2hst52,None,I just watched another episode..I have succumb...,KDRAMA,1598128927,1.0,user_d070b4c03ab0,None,NaN
1940035,Boys Over Flowers,ids296,comment,g2jawam,t3_ids296,None,American television production company should ...,KDRAMA,1598142646,1.0,user_d95e6132f25d,None,NaN
1940036,Boys Over Flowers,ids296,comment,g2mmrc9,t3_ids296,None,**Into the Ring** is over! It gets a 10/10 out...,KDRAMA,1598219289,1.0,user_cc9583b41e94,None,NaN
1940037,Boys Over Flowers,ids296,comment,g2qb5i5,t1_g2mmrc9,None,I enjoyed Let's Eat 2 more than its predecesso...,KDRAMA,1598301997,1.0,user_7fe6c0b7e50a,None,NaN
1940038,Boys Over Flowers,ids296,comment,g2qxm0g,t1_g2fg0n9,None,Definitely not as good as the original but I d...,KDRAMA,1598313564,2.0,user_a104f72bd564,None,NaN


In [35]:
df.to_csv("all_kdrama_threads.csv", index=False) #save the master dataset

Get some descriptive numbers about the resulting full master dataset:

In [36]:
print("Dataset dimensions:", df.shape)

print("\nRecords by drama and type:")
print(pd.crosstab(df["drama"], df["record_type"]))

Dataset dimensions: (1940039, 13)

Records by drama and type:
record_type              comment  submission
drama                                       
Birth Of A Beauty          77754        1033
Boys Over Flowers         631907        6881
Mask Girl                 140716        1533
My Id Is Gangnam Beauty   306718        3628
True Beauty               760762        9107
